In [2]:
import pandas as pd

In [3]:
data=pd.read_csv("india_2000_2024_daily_weather.csv") # load the dataset 
print(data.head()) 

    city        date  temperature_2m_max  temperature_2m_min  \
0  Delhi  2000-01-01                19.9                 7.4   
1  Delhi  2000-01-02                20.0                 5.5   
2  Delhi  2000-01-03                20.1                 6.3   
3  Delhi  2000-01-04                19.8                 6.4   
4  Delhi  2000-01-05                19.4                 5.3   

   apparent_temperature_max  apparent_temperature_min  precipitation_sum  \
0                      19.2                       5.8                0.0   
1                      18.9                       3.4                0.0   
2                      18.2                       4.3                0.0   
3                      18.5                       4.2                0.0   
4                      17.7                       3.2                0.0   

   rain_sum  weather_code  wind_speed_10m_max  wind_gusts_10m_max  \
0       0.0             0                10.9                13.3   
1       0.0         

In [4]:
# removing the duplicate rows if present 

data=data.drop_duplicates()


In [5]:
print(data.isnull().sum()) 
# from the output we can confirm that there are no missing values

city                           0
date                           0
temperature_2m_max             0
temperature_2m_min             0
apparent_temperature_max       0
apparent_temperature_min       0
precipitation_sum              0
rain_sum                       0
weather_code                   0
wind_speed_10m_max             0
wind_gusts_10m_max             0
wind_direction_10m_dominant    0
dtype: int64


In [6]:
print(data.describe()) 
# get the statistical data of all the numerical columns  

       temperature_2m_max  temperature_2m_min  apparent_temperature_max  \
count        91320.000000        91320.000000              91320.000000   
mean            30.900903           21.129450                 33.058268   
std              4.770039            5.291931                  5.988028   
min             12.300000            1.700000                 10.200000   
25%             27.800000           18.000000                 29.200000   
50%             30.700000           22.100000                 33.600000   
75%             33.800000           25.300000                 37.600000   
max             46.400000           34.400000                 49.400000   

       apparent_temperature_min  precipitation_sum      rain_sum  \
count              91320.000000       91320.000000  91320.000000   
mean                  22.863710           3.092751      3.092751   
std                    7.331756           9.164448      9.164448   
min                   -1.100000           0.000000  

In [7]:
# convert string date to pandas date object 

data["date"]=pd.to_datetime(data['date'],format="%Y-%m-%d") 


In [8]:
# sort the values for easier time series analysis  

data=data.sort_values(by=["date","city"],ignore_index=True) 


In [9]:
# extract deeper features like month and year  

data["month"]=data['date'].dt.month
data["year"]=data["date"].dt.year 
print(data["month"].head(5))



0    1
1    1
2    1
3    1
4    1
Name: month, dtype: int32


In [10]:
# creating a season column from the extracted date column  

def get_indian_season(month):

    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Summer'
    elif month in [6, 7, 8, 9]:
        return 'Monsoon'
    else:
        return 'Post-Monsoon'

data['season']=data['month'].apply(get_indian_season) 

print(data["season"].head(5)) 


0    Winter
1    Winter
2    Winter
3    Winter
4    Winter
Name: season, dtype: object


In [11]:
# in the data set both the columns rain_sum and the precipitation indicate the same metric
# i,e amount of rain_fall which is redundant do drop the column 

data=data.drop(columns=["rain_sum"]) 
data["precipitation_log"]=data["precipitation_sum"]


In [12]:
# Map weather_code to standard WMO weather interpretation labels

weather_code_labels = {
    0: "Clear sky",
    1: "Mainly clear",
    2: "Partly cloudy",
    3: "Overcast",
    45: "Fog",
    48: "Depositing rime fog",
    51: "Light drizzle",
    53: "Moderate drizzle",
    55: "Dense drizzle",
    61: "Slight rain",
    63: "Moderate rain",
    65: "Heavy rain",
    71: "Slight snowfall",
    73: "Moderate snowfall",
    75: "Heavy snowfall",
    80: "Slight rain showers",
    81: "Moderate rain showers",
    82: "Violent rain showers",
    95: "Thunderstorm",
    96: "Thunderstorm with slight hail",
    99: "Thunderstorm with heavy hail"
}

data["weather_label"] = data["weather_code"].map(weather_code_labels)

print(data[["weather_code", "weather_label"]].drop_duplicates().sort_values("weather_code"))
print("\nUnmapped codes:", data["weather_label"].isna().sum())

      weather_code     weather_label
0                0         Clear sky
21               1      Mainly clear
2                2     Partly cloudy
1                3          Overcast
42              51     Light drizzle
91              53  Moderate drizzle
245             55     Dense drizzle
117             61       Slight rain
113             63     Moderate rain
1146            65        Heavy rain

Unmapped codes: 0


In [13]:
data.to_csv("preprocessed.csv") # save the preprocessed data